# Poisson in 2D, and relaxation

The Poisson equation −Δu = f is the model problem of numerical PDEs. Its
5-point discretization on an n × n grid is a sparse symmetric positive
definite matrix with five nonzeros per row. This notebook solves it with
Krylov methods and checks the answer against an exact solution. Then it turns
to the oldest iterative methods, Jacobi, Gauss–Seidel and SOR, measures their
convergence against the theory, and shows the one thing they do well, which
is what multigrid builds on.

**Uses:** `mtl5.generators.poisson2d`, `mtl5.generators.laplacian_1d`,
`mtl5.sparse.cg`, `mtl5.sparse.gmres`, `mtl5.mg.smooth`.

**From C++:** [`phase04_sparse_assembly/laplacian_2d.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase04_sparse_assembly/laplacian_2d.cpp), [`phase04_sparse_assembly/smoother_convergence.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase04_sparse_assembly/smoother_convergence.cpp).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mtl5
import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

## The matrix

`mtl5.generators.poisson2d(n, n)` is the 5-point stencil on the n × n
interior points of the unit square, scaled by 1/h² with h = 1 / (n + 1): 4/h²
on the diagonal and −1/h² for each of the four neighbours. Unknown
(ix, iy) is row ix + n·iy, so the neighbours left and right are one column
away and those above and below are n columns away. The C++ example assembles
the same matrix entry by entry with an inserter.

In [ ]:
n = 10
h = 1.0 / (n + 1)
A = mtl5.generators.poisson2d(n, n)
indptr, cols, vals = A.to_csr_arrays()
print(f"{A.shape[0]} unknowns, {A.nnz} nonzeros, at most {np.diff(indptr).max()} per row")
row = 2 + n * 3  # the interior point (ix, iy) = (2, 3)
print(
    "row",
    row,
    ": columns",
    cols[indptr[row] : indptr[row + 1]],
    "values x h^2",
    np.round(vals[indptr[row] : indptr[row + 1]] * h**2, 12),
)
assert A.nnz == 5 * n * n - 4 * n

## Checking against an exact solution

Choose u(x, y) = sin(πx) sin(πy), so f = −Δu = 2π² u. The discrete solution
should match u up to an O(h²) discretization error. All three solvers reach
it in a single iteration, and that is not a coincidence: sampled on the grid,
u is an eigenvector of the discrete Laplacian, so f is too, and a Krylov
method started from a multiple of an eigenvector has nothing left to find
after one step.

In [ ]:
def grid(m):
    xs = np.arange(1, m + 1) / (m + 1)
    X, Y = np.meshgrid(xs, xs)  # rows are y, columns x: row-major is ix + m * iy
    return X, Y


errors, sizes = [], [10, 20, 40, 80]
for m in sizes:
    X, Y = grid(m)
    u = (np.sin(np.pi * X) * np.sin(np.pi * Y)).ravel()
    Am = mtl5.generators.poisson2d(m, m)
    for name, solve in [("cg", ms.cg), ("gmres", ms.gmres)]:
        um, info, report = solve(Am, 2 * np.pi**2 * u, rtol=1e-10, full_output=True)
        assert info == 0 and report.iterations == 1, (m, name)
    errors.append(np.abs(um - u).max())
errors = np.array(errors)
ratios = errors[:-1] / errors[1:]
print(
    "max error:",
    ", ".join(f"{e:.2e}" for e in errors),
    " ratio per halving of h:",
    np.round(ratios, 2),
)
assert (ratios > 3.5).all()  # second order: about 4 per halving

## A harder right-hand side: f = 1

With f = 1, as in the C++ example, every eigenmode is present, and the
solvers have work to do. GMRES keeps every search direction until it
restarts. GMRES(5) restarts every 5 steps and forgets what it has learned,
GMRES(20) every 20. CG needs no restart: for a symmetric matrix a short
recurrence carries all the information GMRES stores.

In [ ]:
results = {}
for m in (10, 30):
    Am = mtl5.generators.poisson2d(m, m)
    bm = np.ones(m * m)
    for label, solve, kw in [
        ("GMRES(5)", ms.gmres, {"restart": 5}),
        ("GMRES(20)", ms.gmres, {"restart": 20}),
        ("CG", ms.cg, {}),
    ]:
        um, info, report = solve(Am, bm, rtol=1e-8, maxiter=5000, full_output=True, **kw)
        assert info == 0, (m, label)
        results[m, label] = (um, report)
    counts = {label: results[m, label][1].iterations for label in ("GMRES(5)", "GMRES(20)", "CG")}
    print(f"{m} x {m} grid:", counts)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.2, 3.7))
for color, label in zip(SERIES, ("GMRES(5)", "GMRES(20)", "CG")):
    report = results[30, label][1]
    ax1.semilogy(report.history / report.history[0], color=color, label=label)
ax1.set_xlabel("iteration")
ax1.set_ylabel("residual / ||b||")
ax1.set_title("30 x 30 grid, f = 1", fontsize=10)
ax1.legend(frameon=False)
ax1.grid(True)
u30 = results[30, "CG"][0].reshape(30, 30)
im = ax2.imshow(u30, origin="lower", extent=(0, 1, 0, 1), cmap="viridis")
fig.colorbar(im, ax=ax2, shrink=0.85)
ax2.set_title("u for f = 1", fontsize=10)
ax2.set_xlabel("x")
ax2.set_ylabel("y")
plt.tight_layout()
plt.show()

for m in (10, 30):
    it = {label: results[m, label][1].iterations for label in ("GMRES(5)", "GMRES(20)", "CG")}
    assert it["CG"] <= it["GMRES(20)"] < it["GMRES(5)"], (m, it)
    # all three reach the same solution
    ref = results[m, "CG"][0]
    for label in ("GMRES(5)", "GMRES(20)"):
        assert np.abs(results[m, label][0] - ref).max() <= 1e-6 * np.abs(ref).max(), (m, label)

The restart length is a trade between memory and convergence. GMRES(5) stores
6 vectors and takes 13 times as many iterations as CG on the 30 × 30 grid.
GMRES(20) stores 21 and takes about 3 times as many. On a symmetric positive
definite matrix CG is the better choice on both counts.

## Relaxation: Jacobi, Gauss–Seidel and SOR

Before Krylov methods there were stationary iterations, each sweep updating
every unknown from its neighbours:

- **Jacobi** updates all unknowns from the old values.
- **Gauss–Seidel** uses each new value as soon as it has it.
- **SOR** moves each Gauss–Seidel update further, by a factor ω, so ω = 1 is
  Gauss–Seidel.

`mtl5.mg.smooth` runs any of them. The C++ example applies them to the 1D
Poisson matrix with 50 unknowns and f = 1, and so does this notebook. The
theory for this matrix is exact. Jacobi's error shrinks by ρ_J = cos(πh) per
sweep, Gauss–Seidel's by ρ_J², and SOR's, at the best ω_opt = 2 / (1 + sin πh),
by about ω_opt − 1.

In [ ]:
n1 = 50
h1 = 1.0 / (n1 + 1)
A1 = ms.from_scipy(ms.to_scipy(mtl5.generators.laplacian_1d(n1)) / h1**2)
b1 = np.ones(n1)
A1_np = ms.to_scipy(A1)
rho_j = np.cos(np.pi * h1)
omega_opt = 2.0 / (1.0 + np.sin(np.pi * h1))


def residual_history(kind, omega=1.0, sweeps=100):
    x = mtl5.vector(np.zeros(n1))
    out = [1.0]
    for _ in range(sweeps):
        x = mtl5.mg.smooth(A1, x, mtl5.vector(b1), kind=kind, sweeps=1, omega=omega)
        out.append(np.linalg.norm(b1 - A1_np @ x.to_numpy()) / np.linalg.norm(b1))
    return np.array(out)


runs = {
    "Jacobi": residual_history("jacobi"),
    "Gauss–Seidel": residual_history("gauss_seidel"),
    "SOR ω = 0.5": residual_history("sor", 0.5),
    "SOR ω = 1": residual_history("sor", 1.0),
    "SOR ω = 1.5": residual_history("sor", 1.5),
    f"SOR ω_opt = {omega_opt:.3f}": residual_history("sor", omega_opt),
}

fig, ax = plt.subplots(figsize=(6.0, 3.8))
colors = SERIES + ["#8a5cd6", "#52514e"]
for color, (label, hist) in zip(colors, runs.items()):
    style = ":" if label == "SOR ω = 1" else "-"
    ax.semilogy(hist, style, color=color, label=label)
ax.set_xlabel("sweep")
ax.set_ylabel("residual / ||b||")
ax.set_title("1D Poisson, n = 50: 100 relaxation sweeps", fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

for label, hist in runs.items():
    print(f"{label:>20}: residual after 100 sweeps {hist[-1]:.2e}")
assert np.array_equal(runs["SOR ω = 1"], runs["Gauss–Seidel"])  # the same iteration, bit for bit

After 100 sweeps Jacobi and Gauss–Seidel have reduced the residual by less
than half. SOR with ω_opt has gained more than three digits. The SOR ω = 1
curve lies exactly on Gauss–Seidel's.

The asymptotic rates can be measured precisely with longer runs. Take the
residual after 2000 and 3000 sweeps; the thousandth root of their ratio is the
contraction per sweep.

In [ ]:
def rate(kind, omega=1.0, start=2000, span=1000):
    x = mtl5.mg.smooth(
        A1, mtl5.vector(np.zeros(n1)), mtl5.vector(b1), kind=kind, sweeps=start, omega=omega
    )
    r1 = np.linalg.norm(b1 - A1_np @ x.to_numpy())
    x = mtl5.mg.smooth(A1, x, mtl5.vector(b1), kind=kind, sweeps=span, omega=omega)
    return (np.linalg.norm(b1 - A1_np @ x.to_numpy()) / r1) ** (1.0 / span)


jac, gs = rate("jacobi"), rate("gauss_seidel")
sor = (runs[f"SOR ω_opt = {omega_opt:.3f}"][60] / runs[f"SOR ω_opt = {omega_opt:.3f}"][20]) ** (
    1 / 40
)
print(f"Jacobi        measured {jac:.10f}   cos(pi h)   = {rho_j:.10f}")
print(f"Gauss-Seidel  measured {gs:.10f}   cos^2(pi h) = {rho_j**2:.10f}")
print(f"SOR(opt)      measured {sor:.4f} (sweeps 20 to 60)   omega_opt - 1 = {omega_opt - 1:.4f}")

assert abs(jac - rho_j) < 1e-8
assert abs(gs - rho_j**2) < 1e-8
assert omega_opt - 1 < sor < 0.95  # near omega_opt - 1, and far below Gauss-Seidel's 0.996

Jacobi and Gauss–Seidel match the theory to every printed digit. SOR at ω_opt
converges a little more slowly than ω_opt − 1 suggests. At exactly ω_opt its
iteration matrix has a repeated eigenvalue without a full set of
eigenvectors, so the error decays like k·ρᵏ rather than ρᵏ.

Even optimal SOR needs O(n) sweeps, and Jacobi and Gauss–Seidel O(n²):
ρ_J = cos(πh) ≈ 1 − π²h²/2. As standalone solvers they are hopeless on fine
grids.

## What relaxation is good at

Relaxation damps some errors far faster than others. Start Gauss–Seidel on
A e = 0, whose solution is e = 0, from an error made of the smoothest mode
sin(πx) and a rough one, sin(48πx), on 63 points. After 5 sweeps the rough
mode is gone and the smooth one has barely changed.

In [ ]:
m = 63
hm = 1.0 / (m + 1)
Am = ms.from_scipy(ms.to_scipy(mtl5.generators.laplacian_1d(m)) / hm**2)
xm = hm * np.arange(1, m + 1)
smooth_mode, rough_mode = np.sin(np.pi * xm), np.sin(48 * np.pi * xm)


def amplitude(e, mode):
    return (e @ mode) / (mode @ mode)


fig, ax = plt.subplots(figsize=(6.0, 3.4))
e = smooth_mode + rough_mode
ax.plot(xm, e, color=INK_2, linewidth=0.8, label="start")
for color, sweeps in zip(SERIES, (1, 5)):
    e_k = mtl5.mg.smooth(
        Am, mtl5.vector(e.copy()), mtl5.vector(np.zeros(m)), sweeps=sweeps
    ).to_numpy()
    ax.plot(
        xm, e_k, color=color, label=f"after {sweeps} Gauss–Seidel sweep{'s' if sweeps > 1 else ''}"
    )
ax.set_xlabel("x")
ax.set_ylabel("error")
ax.set_title("Relaxation removes rough error, not smooth error", fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

e5 = mtl5.mg.smooth(Am, mtl5.vector(e.copy()), mtl5.vector(np.zeros(m)), sweeps=5).to_numpy()
a_smooth, a_rough = amplitude(e5, smooth_mode), amplitude(e5, rough_mode)
print(f"after 5 sweeps: smooth mode {a_smooth:.3f}, rough mode {a_rough:.1e}")
assert amplitude(e5, smooth_mode) > 0.9
assert abs(amplitude(e5, rough_mode)) < 0.01

That is the idea of multigrid. A few sweeps leave only smooth error, and a
smooth error can be represented on a grid with half the points, where it
looks rougher and relaxation works again. The
[multigrid notebook](multigrid.ipynb) builds the hierarchy.

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `inserter<compressed2D<double>> ins(A, 5); ins[r][c] << v;` | `mtl5.generators.poisson2d(n, n)`, assembled by MTL5 |
| `itl::gmres(A, x, b, pc, iter, restart)` | `ms.gmres(A, b, restart=...)` |
| `itl::cg(A, x, b, pc, iter)` | `ms.cg(A, b)` |
| `cyclic_iteration<double> iter(b, 500, 1e-8, 10)` | `full_output=True`, `report.history` |
| `itl::smoother::jacobi<...> jac(A); jac(x, b);` | `mtl5.mg.smooth(A, x, b, kind="jacobi")` |
| `itl::smoother::gauss_seidel<...>`, `itl::smoother::sor<...>(A, omega)` | `kind="gauss_seidel"`, `kind="sor", omega=...` |

The C++ Laplacian example states the exact solution sin(πx) sin(πy) but
solves with f = 1, for which it isn't the solution. This notebook does both:
the exact solution to check accuracy, and f = 1 to compare the solvers.